# Day 56: AI Capstone: Choose and Scope

**Today's question:** how do you turn *"an AI that helps students with money"* into a plan you can finish in
four sessions **and prove** that it works?

We scope one sample idea together, a **Spending Summary Assistant**, from vague to concrete. It is not one of
the capstone options; it is our worked example for the whole week (Days 57–59 build it).

Before running a cell marked 🤔, predict what it will show.

## 1. From vague to concrete

A vague idea cannot be built, tested or graded. Five questions make it concrete:

| Question | Vague answer | Concrete answer |
|----------|--------------|-----------------|
| Who is the user? | "students" | a student who already logs expenses in a CSV file |
| What goes in? | "their money stuff" | the expenses file + a month such as `2026-09` |
| What comes out? | "advice" | a totals table + a 2-sentence summary + 2 saving tips |
| Which part needs AI, and why? | "all of it" | only the friendly wording of the summary and tips |
| How will we know it works? | "it looks good" | measurable success criteria (section 3) |

In [ ]:
vague_idea = "An AI app that helps students with money."

concrete_idea = {
    "user": "a student who logs expenses in a CSV file",
    "input": "the expenses file and a month, e.g. 2026-09",
    "output": "totals table + 2-sentence summary + 2 saving tips",
    "ai_part": "writing the summary and tips in friendly words",
    "plain_python": "loading the CSV and adding up the totals",
}
for key, value in concrete_idea.items():
    print(f"{key:<13} {value}")

🤔 **Predict:** which line of `concrete_idea` could you check with an `assert`, and which could you not?

The *output* and *plain_python* parts are checkable (`assert totals["rent"] == 650.0`).
"Friendly words" is not checkable yet: we fix that with success criteria in section 3.

## 2. Look at the data before promising anything

In [ ]:
from pathlib import Path

import pandas as pd

expenses = pd.read_csv(Path("../../../datasets/expenses.csv"))
print(expenses.shape)
expenses.head(3)

In [ ]:
# .str[:7] keeps the first 7 characters of each date: "2026-09-01" -> "2026-09"
expenses["month"] = expenses["date"].str[:7]
expenses["month"].value_counts().sort_index()

🤔 **Predict:** do we need an AI model to know how much was spent on each category in September?

In [ ]:
september = expenses[expenses["month"] == "2026-09"]
september_totals = september.groupby("category")["amount"].sum().round(2).sort_values(ascending=False)
september_totals

No AI needed: pandas does it exactly, for free, in milliseconds.

**Design rule for the capstone:** numbers come from your code; the model only adds what code cannot do well
(here: friendly wording). An LLM asked to add up 50 amounts may simply get the sum wrong.

## 3. Success criteria you can measure

A success criterion needs **a number** and **a way to measure it**.

| Vague | Measurable |
|-------|------------|
| "the totals are right" | totals for September match a hand calculation in a pytest test |
| "the summary is good" | summary ≤ 30 words and names the biggest flexible category, in ≥ 4 of 5 test months/cases |
| "it is reliable" | 0 crashes on 8 failure cases (bad month, empty file, invalid JSON reply, refusal, no key…) |
| "it works offline" | `uv run pytest -q` passes with no API key; the app runs with the mock |

In [ ]:
def within_word_limit(text, max_words):
    """Success check: the text is not longer than max_words words."""
    return len(text.split()) <= max_words


def mentions(text, word):
    """Success check: the text mentions word (ignoring upper/lower case)."""
    return word.lower() in text.lower()

We can measure a criterion **before any AI exists**, on example outputs we write ourselves. These three are
hand-written examples, not real model output:

In [ ]:
example_outputs = [
    "You spent 1929.20 in September. Food was your biggest flexible category at 660.87.",
    "Great month! Keep it up.",
    "In September your spending was 1929.20 in total, and the largest share went to food, which at 660.87 "
    "was even more than rent, so cooking at home a few more evenings each week could make a real difference.",
]
passed = 0
for text in example_outputs:
    ok = within_word_limit(text, 30) and mentions(text, "food")
    print(ok, "|", len(text.split()), "words |", text[:50])
    if ok:
        passed += 1
print(f"Pass rate: {passed}/{len(example_outputs)}")

🤔 **Predict:** which example fails, and which check does it fail? (The second fails "mentions food", the third
fails the word limit.) Tomorrow these check functions become pytest tests.

## 4. Data & privacy check

Three questions for every capstone:
1. **What data** does the app use, and where does it come from?
2. **Is any of it personal** (names, contact details, IDs, health, money of a real person)?
3. **What leaves the computer**, i.e. what exactly is sent to the API?

In [ ]:
september[["date", "category", "description", "amount"]].head(4)

Our file is synthetic, but in real life a description could read *"dinner with Omar"* or *"pharmacy, flu
medicine"*. That is personal data about the user **and** about other people.

Compare two ways of giving the model what it needs:

In [ ]:
every_row = september.to_csv(index=False)   # everything, including descriptions
only_totals = september_totals.to_string()   # 7 category totals

print(len(every_row), "characters if we send every row")
print(len(only_totals), "characters if we send only the totals")
print("Rough token estimate (about 4 characters per token):", len(every_row) // 4, "vs", len(only_totals) // 4)

In [ ]:
print(only_totals)

The totals are all the model needs to write a summary. Sending only them is **more private, cheaper and
faster**. This idea is called *data minimisation*: send the least data that does the job.

## 5. Risks and fallbacks

List what could go wrong, score it, and decide the fallback **now**, not during the demo.

In [ ]:
risks = [
    {"risk": "Model invents numbers in the summary", "likelihood": 2, "impact": 3,
     "plan": "check every number against the totals (Day 59)"},
    {"risk": "No API key or no internet during the demo", "likelihood": 2, "impact": 3,
     "plan": "offline mock + COURSE_OFFLINE=1 rehearsal"},
    {"risk": "Reply is not valid JSON", "likelihood": 2, "impact": 2,
     "plan": "json.loads in try/except, fall back to a template"},
    {"risk": "Tips sound like financial advice", "likelihood": 1, "impact": 2,
     "plan": "prompt rule: general budgeting tips only"},
    {"risk": "Run out of time", "likelihood": 2, "impact": 3,
     "plan": "build the Must list first; Could items only if time is left"},
]


def risk_score(risk):
    """Likelihood (1-3) x impact (1-3): 9 = deal with it first."""
    return risk["likelihood"] * risk["impact"]


for risk in sorted(risks, key=risk_score, reverse=True):
    print(f"{risk_score(risk)}  {risk['risk']:<42} -> {risk['plan']}")

## 6. MVP and the time budget

**MVP** (minimum viable product) = the smallest version that is useful and demo-able. Sort features into
**Must / Should / Could**:

| Must | Should | Could |
|------|--------|-------|
| load CSV, monthly totals, table | number check against the totals | compare two months |
| AI summary + 2 tips behind one function, mock first | example runs for 3 months in the README | chart of the totals |
| ≥ 5 pytest tests offline, CLI with input checks | | save the summary to a file |
| README with Responsible AI section, demo script | | |

In [ ]:
# Time we have: 4 build sessions of about 45 minutes, plus about 40 minutes of homework after each
available_minutes = 4 * 45 + 4 * 40

must = {
    "load CSV + totals + tests": 60,
    "CLI loop + input checks": 30,
    "template summary (no AI)": 20,
    "prompt + ask_llm + mock + JSON checks": 60,
    "tests with fake models": 40,
    "failure cases + README + demo script": 60,
}
should = {"number check against the totals": 40, "compare two months": 40}

must_minutes = sum(must.values())
print("available:", available_minutes, "min")
print("Must + Should:", must_minutes + sum(should.values()), "min")

🤔 **Predict:** does everything fit? What should we do about it?

In [ ]:
spare = available_minutes - must_minutes
# :.0% shows a fraction as a percentage with no decimals: 0.206 -> 21%
print("Must only:", must_minutes, "min | spare:", spare, f"min ({spare / available_minutes:.0%})")

Must + Should does **not** fit, and estimates are always too low (set-up, debugging, a bad day). Rule of thumb:
the Must list should fill at most about **80 %** of the time. Should items start only when every Must works and
is tested.

## 7. Common scoping mistake: a plan that the data cannot support

The first draft of this plan promised *"tips per shop"*. Checking the data takes one line:

In [ ]:
expenses["merchant"].value_counts()

`KeyError: 'merchant'`: there is no shop column, so that feature is impossible with this data. Finding this
on Day 56 costs one minute; finding it on Day 59 costs the project.

| Scoping mistake | Better |
|-----------------|--------|
| "The AI calculates the totals" | code calculates, the model words |
| "Success = it works" | numbers + how they are measured |
| "I'll use my classmates' real data" | synthetic or public data, or your own non-sensitive data |
| "I'll train a neural network / build a website" | out of scope: scikit-learn model or LLM + CLI |
| "Everything is a Must" | at most 5 Musts; the rest are Should/Could |

## Recap

1. Make the idea concrete: user, input, output, which part is AI and why.
2. Look at the data **before** promising features.
3. Success criteria have a number and a measurement, and can often be written as small check functions.
4. Data & privacy check: what data, is it personal, what goes to the API. Send the minimum.
5. Score risks, plan fallbacks, and protect the Must list.

**Next time:** build the non-AI core first, with modules and pytest tests.